# TripMe - Generate Historical Significance Field (Google Colab version)

Sources used, in priority order, for each place:
1. **Mahavamsa** (Wilhelm Geiger's 1912 English translation) - the great
   Pali chronicle of Sri Lanka, covering ancient/medieval history.
2. **Dipavamsa** (Hermann Oldenberg's 1879 English translation) - the older,
   shorter chronicle, used as a fallback/supplement.
3. **Wikipedia** - for places not covered by the chronicles (more recent
   sites, colonial-era forts, modern landmarks, etc).

For every place, the notebook searches these sources for a matching
passage, and if found, asks the LLM (Qwen2.5-3B-Instruct) to summarize
ONLY the facts in that passage into a ~60-100 word `historical_significance`
field - it is never allowed to answer from its own memory. If no source
passage is found at all, no field is added (nothing invented).

## Why Colab, and the time budget
This is built for Google Colab's free tier: a session can run for several
hours but is not guaranteed past ~5 hours, so this notebook hard-stops
itself at **4 hours 55 minutes** of wall-clock runtime, saves everything
collected so far, and prints exactly how many places got real sourced
history before you need to download the output and (if there's more left)
start a fresh session to continue.

## How to run this in Colab
1. Open this notebook in Colab (File -> Upload notebook, or open from Drive).
2. Runtime -> Change runtime type -> GPU (T4 is fine).
3. Run all cells. It will prompt you to upload your local `data/raw` folder
   (zipped) the first time - or mount Google Drive if you'd rather keep it
   there permanently (see the upload cell below for both options).
4. When it finishes (or hits the 4h55m limit), the last cell zips the
   output and triggers a browser download automatically - no manual
   "Save Version" step like Kaggle.
5. Copy the downloaded `raw_updated_history/` contents over your local
   `data/raw` folder, then run `01_merge_places.py` to refresh
   `data/processed/places.json`.

This notebook is resumable: progress is checkpointed to
`history_progress.json` inside the output folder. If you have to start a
new Colab session, re-upload that progress file alongside `data/raw` and
it will pick up exactly where it left off instead of re-processing places
(and re-downloading source texts) it already finished.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece tqdm requests
print("Dependencies installed.")

## Get your data/raw folder into this Colab session

Two options - use whichever is easier for you. Only one of the two cells
below needs to actually run.

In [ ]:
# OPTION A: use a zip of your local data/raw folder.
# If you've already uploaded a .zip to the Colab file browser (the folder
# icon on the left - drag-and-drop or the upload button works fine), this
# picks it up automatically. Otherwise it opens an upload dialog.
from pathlib import Path
import zipfile

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RAW_INPUT_DIR = Path("data/raw")


def extract_and_locate(zip_path: Path) -> Path:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall("data_raw_upload")
    # Handle both "data/raw/..." and "raw/..." zip layouts.
    candidates = [p for p in Path("data_raw_upload").rglob("*") if p.is_dir() and p.name == "raw"]
    return candidates[0] if candidates else Path("data_raw_upload")


if IN_COLAB:
    # Look for a zip already sitting in /content (e.g. dragged into the
    # Files sidebar, or uploaded via the sidebar's own upload button)
    # before falling back to the files.upload() dialog.
    already_uploaded = sorted(Path(".").glob("*.zip"))
    if already_uploaded:
        zip_path = already_uploaded[0]
        print(f"Found already-uploaded zip: {zip_path}")
        RAW_INPUT_DIR = extract_and_locate(zip_path)
        print("Using uploaded data at:", RAW_INPUT_DIR)
    else:
        print("No .zip found in the Files sidebar yet - opening an upload dialog "
              "(skip this with Cancel if you're using Option B - Google Drive - instead).")
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith(".zip"):
                RAW_INPUT_DIR = extract_and_locate(Path(fname))
                print("Using uploaded data at:", RAW_INPUT_DIR)

In [ ]:
# OPTION B: mount Google Drive instead (better if you're resuming across
# multiple sessions - keep data/raw and the progress checkpoint in Drive so
# you don't have to re-upload every time). Uncomment and edit the path to
# match where you put it in Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# RAW_INPUT_DIR = Path("/content/drive/MyDrive/tripme/data/raw")
# print("Using Drive data at:", RAW_INPUT_DIR)

In [ ]:
assert RAW_INPUT_DIR.exists(), (
    f"{RAW_INPUT_DIR} does not exist - run Option A or Option B above first."
)

OUTPUT_DIR = Path("raw_updated_history")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_FILE = OUTPUT_DIR / "history_progress.json"

print("Reading from:", RAW_INPUT_DIR)
print("Writing to:", OUTPUT_DIR)

## Load every place record from data/raw

In [ ]:
import json

def load_records(path: Path):
    text = path.read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)
    while idx < n:
        while idx < n and text[idx] in " \t\r\n":
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(text, idx)
        yield obj
        idx = end


all_files = sorted(RAW_INPUT_DIR.rglob("*.jsonl"))
print(f"Found {len(all_files)} .jsonl files under {RAW_INPUT_DIR}")

records_by_file = {}
total_records = 0
for f in all_files:
    recs = list(load_records(f))
    records_by_file[f] = recs
    total_records += len(recs)

print(f"Loaded {total_records} place records total.")

## Download and chunk the Mahavamsa and Dipavamsa

Both are public-domain English translations from archive.org (OCR'd from
scanned books, so expect occasional OCR noise - that's fine, the LLM
summarization step downstream is robust to minor typos). Each is split
into chapter-sized chunks so a place-name match can be traced back to a
specific, reasonably short passage rather than the whole book.

In [ ]:
import re
import time

import requests

CHRONICLE_CACHE_DIR = Path("chronicle_cache")
CHRONICLE_CACHE_DIR.mkdir(exist_ok=True)

MAHAVAMSA_URL = "https://archive.org/download/mahavamsaorgreat00maha/mahavamsaorgreat00maha_djvu.txt"
DIPAVAMSA_URL = "https://archive.org/download/dpavasaanancien01oldegoog/dpavasaanancien01oldegoog_djvu.txt"


def download_text(url: str, cache_name: str) -> str:
    cache_path = CHRONICLE_CACHE_DIR / cache_name
    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    cache_path.write_text(resp.text, encoding="utf-8")
    return resp.text


def chunk_mahavamsa(text: str) -> list[tuple[str, str]]:
    """Splits on 'CHAPTER <roman numeral>' headings. Returns [(label, chunk_text), ...]."""
    parts = re.split(r"\n\s*CHAPTER\s+([IVXLC]+)\s*\n", text)
    # re.split with a capturing group interleaves: [pre, numeral1, body1, numeral2, body2, ...]
    chunks = []
    for i in range(1, len(parts) - 1, 2):
        numeral, body = parts[i], parts[i + 1]
        chunks.append((f"Mahavamsa, Chapter {numeral}", body.strip()))
    return chunks


def chunk_dipavamsa(text: str) -> list[tuple[str, str]]:
    """Skips Oldenberg's long scholarly introduction and starts at the
    actual translation, then splits on Roman-numeral chapter markers."""
    marker = text.find("TRANSLATION.")
    body_text = text[marker:] if marker != -1 else text
    parts = re.split(r"\n\s*([IVXLC]+)\.\s*\n", body_text)
    chunks = []
    for i in range(1, len(parts) - 1, 2):
        numeral, body = parts[i], parts[i + 1]
        if len(body.strip()) > 100:  # skip stray false-positive matches
            chunks.append((f"Dipavamsa, Chapter {numeral}", body.strip()))
    return chunks


print("Downloading Mahavamsa (Geiger, 1912)...")
mahavamsa_text = download_text(MAHAVAMSA_URL, "mahavamsa.txt")
mahavamsa_chunks = chunk_mahavamsa(mahavamsa_text)
print(f"  {len(mahavamsa_text):,} chars, split into {len(mahavamsa_chunks)} chapters.")

print("Downloading Dipavamsa (Oldenberg, 1879)...")
dipavamsa_text = download_text(DIPAVAMSA_URL, "dipavamsa.txt")
dipavamsa_chunks = chunk_dipavamsa(dipavamsa_text)
print(f"  {len(dipavamsa_text):,} chars, split into {len(dipavamsa_chunks)} chapters.")

CHRONICLE_CHUNKS = mahavamsa_chunks + dipavamsa_chunks
print(f"Total chronicle chunks to search: {len(CHRONICLE_CHUNKS)}")

## Matching a place name against the chronicles

Ancient/medieval place names in these translations often use older
transliterations (e.g. "Anuradhapura" appears constantly, but a modern
temple name might appear as a Pali/Sanskrit variant spelling). This does a
straightforward case-insensitive substring match against each chunk, plus
a looser match against just the first significant word of the place name
(many OSM names are "X Temple", "X Vihara", "X Devalaya" etc. where only
"X" appears in the chronicle). This is deliberately simple/precision-first:
a missed match just means falling through to Wikipedia, which is a fine
outcome, whereas a wrong match risks feeding the LLM an irrelevant passage.

In [ ]:
STOPWORD_SUFFIXES = {
    "temple", "vihara", "viharaya", "devalaya", "devale", "kovil", "church",
    "mosque", "falls", "beach", "fort", "museum", "park", "estate", "ruins",
    "dagoba", "stupa", "rock", "cave", "national", "sanctuary", "reservoir",
    "tank", "bridge", "lake", "mountain", "peak", "garden", "gardens",
}


def significant_name_terms(name: str) -> list[str]:
    words = re.findall(r"[A-Za-z]+", name)
    return [w for w in words if len(w) > 3 and w.lower() not in STOPWORD_SUFFIXES]


def find_chronicle_match(rec: dict) -> tuple[str, str] | None:
    name = rec.get("name", "")
    terms = significant_name_terms(name)
    if not terms:
        return None
    for label, chunk_text in CHRONICLE_CHUNKS:
        lowered = chunk_text.lower()
        if any(term.lower() in lowered for term in terms):
            return label, chunk_text
    return None


# Smoke test - Anuradhapura-related terms should hit something; a made-up
# name should not.
print("Anuradhapura match:", (lambda m: m[0] if m else None)(
    find_chronicle_match({"name": "Anuradhapura Ruins"})
))
print("Nonsense match:", find_chronicle_match({"name": "Zzqx Made Up Falls Xyzabc"}))

## Wikipedia lookup (fallback for places the chronicles don't cover)

In [ ]:
WIKI_SESSION = requests.Session()
WIKI_SESSION.headers.update({
    "User-Agent": "TripMe-SriLanka-DataPipeline/1.0 (educational travel-app dataset project)"
})
WIKI_REQUEST_DELAY_SEC = 0.2
MIN_EXTRACT_CHARS = 200


def wikipedia_search_title(query: str) -> str | None:
    try:
        resp = WIKI_SESSION.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "list": "search", "srsearch": query, "srlimit": 1, "format": "json"},
            timeout=15,
        )
        resp.raise_for_status()
        results = resp.json().get("query", {}).get("search", [])
        return results[0]["title"] if results else None
    except Exception:
        return None


def wikipedia_fetch_extract(title: str) -> str | None:
    try:
        resp = WIKI_SESSION.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "prop": "extracts", "explaintext": 1, "titles": title, "format": "json"},
            timeout=15,
        )
        resp.raise_for_status()
        pages = resp.json().get("query", {}).get("pages", {})
        for page in pages.values():
            extract = page.get("extract", "")
            if extract and "may refer to" not in extract[:200].lower():
                return extract
        return None
    except Exception:
        return None


def find_wikipedia_article(rec: dict) -> tuple[str, str] | None:
    name = rec.get("name", "").strip()
    district = rec.get("district_id", "").strip()
    if not name:
        return None
    for query in (f"{name} {district} Sri Lanka", f"{name} Sri Lanka"):
        title = wikipedia_search_title(query)
        time.sleep(WIKI_REQUEST_DELAY_SEC)
        if not title:
            continue
        extract = wikipedia_fetch_extract(title)
        time.sleep(WIKI_REQUEST_DELAY_SEC)
        if extract and len(extract) >= MIN_EXTRACT_CHARS:
            return f"Wikipedia: {title}", extract
    return None


def find_source_passage(rec: dict) -> tuple[str, str] | None:
    """Chronicles first (older, more authoritative for ancient sites),
    Wikipedia as fallback for places the chronicles don't mention."""
    return find_chronicle_match(rec) or find_wikipedia_article(rec)

## Load the LLM (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime -> Change runtime type -> GPU (T4), then "
        "Runtime -> Restart and run all."
    )

print(f"Loading {LLM_MODEL_NAME} (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME, quantization_config=bnb_config, device_map="auto")
llm_model.eval()
llm_tokenizer.padding_side = "left"
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
print("LLM loaded and ready.")

## Summarize the matched passage (grounded - LLM may only use the given text)

In [ ]:
NO_HISTORY_MARKER = "NO_HISTORY_IN_TEXT"

HISTORY_SYSTEM_PROMPT = (
    "You summarize historical source text into a short historical/cultural "
    "significance blurb for a Sri Lanka travel app. You must use ONLY the "
    "facts present in the source text you are given below - do not add any "
    "fact, date, name, or claim that is not stated in that text, even if you "
    "think you know more about the place from elsewhere. The source may be "
    "from an old chronicle (Mahavamsa/Dipavamsa) with archaic phrasing, or a "
    "modern Wikipedia article - either way, work only from what's given. If "
    "the text given to you does not actually mention this specific place's "
    "history or significance, reply with exactly the single token "
    f"{NO_HISTORY_MARKER} and nothing else. Otherwise, write 60-100 words, "
    "one paragraph, no markdown, no preamble, no meta-commentary about the "
    "source - just the historical/cultural content itself, in your own words."
)

MAX_SOURCE_CHARS = 3000


def build_history_prompt(rec: dict, source_label: str, source_text: str) -> str:
    name = rec.get("name", "")
    truncated = source_text[:MAX_SOURCE_CHARS]
    return (
        f'Place name: "{name}"\n'
        f'Source: {source_label}\n\n'
        f"Source text:\n{truncated}\n\n"
        "Does this text mention this specific place's historical/cultural "
        "significance? If yes, summarize it in 60-100 words using only "
        f"facts stated above. If no, reply with exactly {NO_HISTORY_MARKER}."
    )


BATCH_SIZE = 16


def llm_generate_batch(prompts: list[str], max_new_tokens=170, temperature=0.2) -> list[str]:
    formatted = [
        llm_tokenizer.apply_chat_template(
            [{"role": "system", "content": HISTORY_SYSTEM_PROMPT}, {"role": "user", "content": p}],
            add_generation_prompt=True, tokenize=False,
        )
        for p in prompts
    ]
    inputs = llm_tokenizer(formatted, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=True, pad_token_id=llm_tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    texts = llm_tokenizer.batch_decode(out[:, input_len:], skip_special_tokens=True)
    return [t.strip() for t in texts]


def is_valid_history(text: str) -> bool:
    if not text:
        return False
    return 40 <= len(text.split()) <= 130


def summarize_batch(items: list[tuple[dict, str, str]], max_retries=2) -> dict:
    results = {}
    remaining = items
    for attempt in range(max_retries + 1):
        if not remaining:
            break
        prompts = [build_history_prompt(rec, label, text) for rec, label, text in remaining]
        try:
            texts = llm_generate_batch(prompts)
        except Exception as e:
            print(f"  batch generation error (attempt {attempt+1}): {e}")
            continue
        still_remaining = []
        for (rec, label, src_text), out_text in zip(remaining, texts):
            if NO_HISTORY_MARKER in out_text:
                results[rec["id"]] = None
            elif is_valid_history(out_text):
                results[rec["id"]] = {"text": out_text, "source": label}
            else:
                still_remaining.append((rec, label, src_text))
        remaining = still_remaining
    for rec, _, _ in remaining:
        results[rec["id"]] = None
    return results

## Process every record: source lookup, then batched LLM summarization

Hard-stops at 4h55m so there's always time left to save and download
before a Colab session might get reclaimed.

In [ ]:
import time

from tqdm.auto import tqdm

MAX_RUNTIME_HOURS = 4 + 55 / 60  # 4h55m
run_deadline = time.time() + MAX_RUNTIME_HOURS * 3600


def load_progress() -> dict:
    if PROGRESS_FILE.exists():
        return json.loads(PROGRESS_FILE.read_text(encoding="utf-8"))
    return {"done": {}}  # id -> {"text":..., "source":...} or null


def save_progress(progress: dict) -> None:
    PROGRESS_FILE.write_text(json.dumps(progress, ensure_ascii=False), encoding="utf-8")


progress = load_progress()
done_map = progress["done"]
print(f"Resuming: {len(done_map)} places already processed in a previous run.")

all_records = [rec for recs in records_by_file.values() for rec in recs]
pending = [rec for rec in all_records if rec.get("id") not in done_map]
found_so_far = sum(1 for v in done_map.values() if v)
print(f"{found_so_far} places already have real sourced history.")
print(f"{len(done_map) - found_so_far} places already checked, nothing found.")
print(f"{len(pending)} of {len(all_records)} records still need processing.")
print(f"Time budget: {MAX_RUNTIME_HOURS:.2f} hours from now.")

stopped_early = False
source_batch = []  # list of (record, source_label, source_text) waiting to be summarized
chronicle_hits = 0
wikipedia_hits = 0
no_source_found = 0


def flush_batch():
    global done_map
    if not source_batch:
        return
    results = summarize_batch(list(source_batch))
    done_map.update(results)
    source_batch.clear()
    save_progress(progress)


last_progress_print = time.time()
for i, rec in enumerate(tqdm(pending, desc="Finding sources + summarizing")):
    if time.time() >= run_deadline:
        stopped_early = True
        print(f"\nHit the {MAX_RUNTIME_HOURS:.2f}-hour safety limit - stopping and saving progress.")
        break

    found = find_source_passage(rec)
    if found is None:
        done_map[rec["id"]] = None
        no_source_found += 1
        continue

    label, text = found
    if label.startswith("Wikipedia"):
        wikipedia_hits += 1
    else:
        chronicle_hits += 1
    source_batch.append((rec, label, text))
    if len(source_batch) >= BATCH_SIZE:
        flush_batch()

    # Periodic knowledge-coverage status update, since this can run for hours.
    if time.time() - last_progress_print > 60:  # every 1 minute
        elapsed_min = (time.time() - (run_deadline - MAX_RUNTIME_HOURS * 3600)) / 60
        found_now = sum(1 for v in done_map.values() if v)
        print(f"\n[{elapsed_min:.0f} min elapsed] processed {i+1}/{len(pending)} pending "
              f"this session | {found_now} total places now have real history "
              f"({chronicle_hits} from chronicles, {wikipedia_hits} from Wikipedia so far this session)")
        last_progress_print = time.time()

flush_batch()
save_progress(progress)

found_count = sum(1 for v in done_map.values() if v)
print(f"\n=== Knowledge coverage summary ===")
print(f"Total places in dataset: {len(all_records)}")
print(f"Places with real, sourced historical_significance: {found_count} "
      f"({found_count / len(all_records) * 100:.1f}% of all places)")
print(f"  - from Mahavamsa/Dipavamsa (this session): {chronicle_hits}")
print(f"  - from Wikipedia (this session): {wikipedia_hits}")
print(f"Places checked with no source found: {len(done_map) - found_count}")
print(f"Still unprocessed: {len(all_records) - len(done_map)}"
      + (" - hit the time limit, run again to continue." if stopped_early
         else " - re-run this cell to retry any that errored out."))

## Write updated records back out, same per-district/per-category layout

In [ ]:
added_count = 0
unchanged_count = 0

for src_path, recs in records_by_file.items():
    rel = src_path.relative_to(RAW_INPUT_DIR)
    out_path = OUTPUT_DIR / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in recs:
            rid = rec.get("id")
            entry = done_map.get(rid)
            if entry:
                rec = dict(rec)
                rec["historical_significance"] = entry["text"]
                rec["historical_significance_source"] = entry["source"]
                added_count += 1
            else:
                unchanged_count += 1
            f.write(json.dumps(rec, indent=4, ensure_ascii=False))
            f.write("\n\n")

print(f"Wrote {len(records_by_file)} files to {OUTPUT_DIR}")
print(f"{added_count} records got a real, sourced historical_significance field.")
print(f"{unchanged_count} records had no matching source - left unchanged.")

## Download the result to your laptop

Zips the output folder (place records + progress checkpoint) and triggers
a browser download automatically - no "Save Version" step needed like on
Kaggle. If you're not on the free tier's time limit and want to keep
going, you can also just leave this session running and re-run the
processing cell later instead of downloading now.

In [ ]:
import shutil

zip_path = shutil.make_archive("raw_updated_history", "zip", OUTPUT_DIR)
print(f"Zipped to {zip_path}")

if IN_COLAB:
    files.download(zip_path)
    print("Download triggered - check your browser's downloads.")
else:
    print(f"Not running in Colab - find your zip at: {zip_path}")

print("\nNext step: unzip this, then copy the contents of raw_updated_history/ "
      "over your local data/raw folder (merges the new fields into your "
      "existing records), then run 01_merge_places.py to refresh "
      "data/processed/places.json.")